<a href="https://colab.research.google.com/github/abbosaliboev/Data_Science/blob/main/3_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Apriori & FP tree

In [1]:
!pip install pip --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [2]:
!pip install mlxtend==0.18

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 137.5 kB/s  0:00:08
  Attempting uninstall: mlxtend
    Found existing installation: mlxtend 0.23.4
    Uninstalling mlxtend-0.23.4:
      Successfully uninstalled mlxtend-0.23.4


In [3]:
import mlxtend
mlxtend.__version__

'0.18.0'

In [4]:
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [5]:
import pandas as pd
import os

## Dataset import 및 확인

In [6]:
from google.colab import drive
drive.mount('/content/drive')

basicpath = '/content/drive/MyDrive/'

Mounted at /content/drive


In [7]:
path = basicpath + 'CBNUDatascience_colab/2-3주차'
file = 'BreadBasket_DMS.csv'
data = pd.read_csv(os.path.join(path, file), index_col = None)

In [8]:
data.head(10)

,Date,Time,Transaction,Item
0,2016.10.30,9:58:11,1,Bread
1,2016.10.30,10:05:34,2,Scandinavian
2,2016.10.30,10:05:34,2,Scandinavian
3,2016.10.30,10:07:57,3,Hot chocolate
4,2016.10.30,10:07:57,3,Jam
5,2016.10.30,10:07:57,3,Cookies
6,2016.10.30,10:08:41,4,Muffin
7,2016.10.30,10:13:03,5,Coffee
8,2016.10.30,10:13:03,5,Pastry
9,2016.10.30,10:13:03,5,Bread


In [9]:
data.shape

(21293, 4)

## 과제1-1 Data Cleaning


위 데이터를 가공하여, transaction과 item이 column이 되는  
다음과 같은 형태의 새로운 DataFrame을 만들어 변수 data에 저장하시오.

 transaction | item
:-------------:|:------:
      0      |   7  
      0      |   14
      1      |   9  
      2      |   18
...  


In [18]:
data['Item'] = data['Item'].str.lower()
data.head()

,Date,Time,Transaction,Item
0,2016.10.30,9:58:11,1,bread
1,2016.10.30,10:05:34,2,scandinavian
2,2016.10.30,10:05:34,2,scandinavian
3,2016.10.30,10:07:57,3,hot chocolate
4,2016.10.30,10:07:57,3,jam


In [17]:
none_indexes = data[data['Item'] == 'none'].index
data = data.drop(none_indexes)
data.head()

,Date,Time,Transaction,Item
0,2016.10.30,9:58:11,1,bread
1,2016.10.30,10:05:34,2,scandinavian
2,2016.10.30,10:05:34,2,scandinavian
3,2016.10.30,10:07:57,3,hot chocolate
4,2016.10.30,10:07:57,3,jam


### Data 간단 분석_item당 등장횟수

In [21]:
data

,Date,Time,Transaction,Item
0,2016.10.30,9:58:11,1,bread
1,2016.10.30,10:05:34,2,scandinavian
2,2016.10.30,10:05:34,2,scandinavian
3,2016.10.30,10:07:57,3,hot chocolate
4,2016.10.30,10:07:57,3,jam
...,...,...,...,...
21288,2017.4.9,14:32:58,9682,coffee
21289,2017.4.9,14:32:58,9682,tea
21290,2017.4.9,14:57:06,9683,coffee
21291,2017.4.9,14:57:06,9683,pastry


In [23]:
len(data['Item'].unique())

95

In [24]:
top10_items = data['Item'].value_counts().head(10)
top10_items

,count
Item,
coffee,5471
bread,3325
tea,1435
cake,1025
pastry,856
sandwich,771
medialuna,616
hot chocolate,590
cookies,540


In [25]:
len(data['Transaction'].unique())

9531

## 과제 1-2 one-hot-encoding vector 만들기
위 data를 가공하여 one-hot-encoding 형태의 데이터를 만들고  
이를 hot_encoded_data 변수에 저장하시오  
이때 one-hot-encoding 형태의 column은 item, row는 transaction이어야 함  

In [26]:
# 1. Group items by transaction and spread them into columns
# This creates a table where rows are transaction IDs  and columns are items
hot_encoded_data = data.groupby(['Transaction', 'Item'])['Item'].count().unstack().reset_index().fillna(0).set_index('Transaction')

# Show the first few rows to see the numbers
hot_encoded_data.head()

Item,adjustment,afternoon with the baker,alfajores,argentina night,art tray,bacon,baguette,bakewell,bare popcorn,basket,...,the bart,the nomad,tiffin,toast,truffles,tshirt,valentine's card,vegan feast,vegan mincepie,victorian sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
# 2. Define a rule to change numbers to 0 or 1
def encode_units(x):
    if x <= 0:
        return 0 # Item not in basket
    if x >= 1:
        return 1 # Item is in basket

# 3. Apply this rule to the whole table
# Use .map() to avoid the "FutureWarning" error
hot_encoded_data = hot_encoded_data.map(encode_units)

# Check the final table
hot_encoded_data.head()

Item,adjustment,afternoon with the baker,alfajores,argentina night,art tray,bacon,baguette,bakewell,bare popcorn,basket,...,the bart,the nomad,tiffin,toast,truffles,tshirt,valentine's card,vegan feast,vegan mincepie,victorian sponge
Transaction,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 과제 1-3
mlxtend의 apriori & fp-growth 라이브러리를 활용하여 위 데이터에서 frequent itemset을 추출하시오.  
이때, min_support는 0.02로 하시오.  
association_rules 라이브러리를 활용하여 추출한 frequent itemset에서 association rule을 만드시오.

In [28]:
# 1.Find items that appear in at least 2%  of all transactions
# We use min_support=0.02 as requested in the  task
frequent_itemsets = apriori(hot_encoded_data, min_support=0.02, use_colnames=True)

# 2. Show  the list of frequent items
frequent_itemsets

,support,itemsets
0,0.036344,(alfajores)
1,0.327205,(bread)
2,0.040042,(brownie)
3,0.103856,(cake)
4,0.478394,(coffee)
5,0.054411,(cookies)
6,0.039197,(farm house)
7,0.058320,(hot chocolate)
8,0.038563,(juice)
9,0.061807,(medialuna)


In [ ]:
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1)

In [30]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
18,(toast),(coffee),0.033597,0.478394,0.023666,0.704403,1.472431,0.007593,1.764582
12,(medialuna),(coffee),0.061807,0.478394,0.035182,0.569231,1.189878,0.005614,1.210871
14,(pastry),(coffee),0.086107,0.478394,0.047544,0.552147,1.154168,0.006351,1.164682
10,(juice),(coffee),0.038563,0.478394,0.020602,0.534247,1.116750,0.002154,1.119919
16,(sandwich),(coffee),0.071844,0.478394,0.038246,0.532353,1.112792,0.003877,1.115384
2,(cake),(coffee),0.103856,0.478394,0.054728,0.526958,1.101515,0.005044,1.102664
6,(cookies),(coffee),0.054411,0.478394,0.028209,0.518447,1.083723,0.002179,1.083174
8,(hot chocolate),(coffee),0.058320,0.478394,0.029583,0.507246,1.060311,0.001683,1.058553
0,(pastry),(bread),0.086107,0.327205,0.029160,0.338650,1.034977,0.000985,1.017305
4,(cake),(tea),0.103856,0.142631,0.023772,0.228891,1.604781,0.008959,1.111865


In [29]:
# 3. Create association rules using the 'lift'  metric
# We want to seee rules where lift is at least 1
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1)

# 4.Sort the rules by 'confidence' to see the strongest connections first
rules = rules.sort_values('confidence', ascending=False)

# 5. Show  the final result
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
18,(toast),(coffee),0.033597,0.478394,0.023666,0.704403,1.472431,0.007593,1.764582
12,(medialuna),(coffee),0.061807,0.478394,0.035182,0.569231,1.189878,0.005614,1.210871
14,(pastry),(coffee),0.086107,0.478394,0.047544,0.552147,1.154168,0.006351,1.164682
10,(juice),(coffee),0.038563,0.478394,0.020602,0.534247,1.116750,0.002154,1.119919
16,(sandwich),(coffee),0.071844,0.478394,0.038246,0.532353,1.112792,0.003877,1.115384
2,(cake),(coffee),0.103856,0.478394,0.054728,0.526958,1.101515,0.005044,1.102664
6,(cookies),(coffee),0.054411,0.478394,0.028209,0.518447,1.083723,0.002179,1.083174
8,(hot chocolate),(coffee),0.058320,0.478394,0.029583,0.507246,1.060311,0.001683,1.058553
0,(pastry),(bread),0.086107,0.327205,0.029160,0.338650,1.034977,0.000985,1.017305
4,(cake),(tea),0.103856,0.142631,0.023772,0.228891,1.604781,0.008959,1.111865


In [31]:
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1)

In [32]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
0,(pastry),(bread),0.086107,0.327205,0.029160,0.338650,1.034977,0.000985,1.017305
1,(bread),(pastry),0.327205,0.086107,0.029160,0.089119,1.034977,0.000985,1.003306
2,(cake),(coffee),0.103856,0.478394,0.054728,0.526958,1.101515,0.005044,1.102664
3,(coffee),(cake),0.478394,0.103856,0.054728,0.114399,1.101515,0.005044,1.011905
4,(cake),(tea),0.103856,0.142631,0.023772,0.228891,1.604781,0.008959,1.111865
5,(tea),(cake),0.142631,0.103856,0.023772,0.166667,1.604781,0.008959,1.075372
6,(cookies),(coffee),0.054411,0.478394,0.028209,0.518447,1.083723,0.002179,1.083174
7,(coffee),(cookies),0.478394,0.054411,0.028209,0.058966,1.083723,0.002179,1.004841
8,(hot chocolate),(coffee),0.058320,0.478394,0.029583,0.507246,1.060311,0.001683,1.058553
9,(coffee),(hot chocolate),0.478394,0.058320,0.029583,0.061837,1.060311,0.001683,1.003749
